# LTX video: app server

**Experimental, not GPU-validated yet.** This turns a Colab GPU session into a small personal generation endpoint for your app's **Video** page — the same workflow as the voice tool: start Colab, paste the printed address into the app, generate, and the MP4 saves straight into your hub. Nothing is downloaded to your PC.

Choose **Runtime → Change runtime type → GPU**, then run every cell in order. The last cell prints a public address and a token; both die with the session. One job runs at a time; extra requests queue. Keep the token private — it is the only thing stopping a stranger from spending your GPU time.

Weights use the [LTXV Open Weights License](https://huggingface.co/Lightricks/LTX-Video/blob/main/LTX-Video-Open-Weights-License-0.X.txt). Use personally, without keep-alive or usage-limit workarounds; free Colab GPU availability is not guaranteed.

In [ ]:
# 1. Check the assigned machine before downloading anything.
import json, os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import Video, display

ROOT = Path('/content/ltx-video-test')
ROOT.mkdir(exist_ok=True)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode:
    raise RuntimeError('No GPU is available. Choose a GPU runtime; do not run this test on CPU.')
print('Assigned GPU:', gpu.stdout.strip())
free_gib = shutil.disk_usage('/content').free / 1024**3
ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
print(f'System RAM: {ram_gib:.1f} GiB; free disk: {free_gib:.1f} GiB')
if free_gib < 40:
    raise RuntimeError('This isolated test needs at least 40 GiB of free disk for dependencies and model files.')
print('These checks do not guarantee model fit. Host RAM and GPU memory can still run out.')

In [ ]:
# 2. Create an isolated Python environment; do not replace Colab's notebook packages.
def run(args, **kwargs):
    subprocess.run([str(a) for a in args], check=True, **kwargs)

run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.8.22'])
UV = [sys.executable, '-m', 'uv']
ENV = ROOT / 'env'
run(UV + ['venv', '--python', '3.11', str(ENV)])
PYTHON = ENV / 'bin/python'
run(UV + ['pip', 'install', '--python', str(PYTHON), 'torch==2.6.0', 'torchvision==0.21.0', '--index-url', 'https://download.pytorch.org/whl/cu124'])
run(UV + ['pip', 'install', '--python', str(PYTHON),
    'diffusers==0.35.1', 'transformers==4.51.3', 'huggingface-hub==0.30.2',
    'accelerate==1.6.0', 'safetensors==0.5.3', 'sentencepiece==0.2.0',
    'numpy==1.26.4', 'Pillow==11.2.1', 'PyYAML==6.0.2', 'einops==0.8.1',
    'timm==1.0.15', 'imageio==2.37.0', 'imageio-ffmpeg==0.6.0', 'av==14.3.0'])

SOURCE_REVISION = '4b2d053057623ddd4d0a1d3e9cd28890e9ef487f'
REPO = ROOT / 'LTX-Video'
if not REPO.exists():
    run(['git', 'clone', '--no-checkout', 'https://github.com/Lightricks/LTX-Video.git', str(REPO)])
run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_REVISION])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if actual != SOURCE_REVISION:
    raise RuntimeError('Source revision does not match the pinned test.')
run(UV + ['pip', 'install', '--python', str(PYTHON), '--no-deps', '-e', str(REPO)])
run(UV + ['pip', 'check', '--python', str(PYTHON)])
resolved = subprocess.check_output(UV + ['pip', 'freeze', '--python', str(PYTHON)], text=True)
(ROOT / 'environment.txt').write_text(resolved)
print('Isolated environment ready. No model has generated anything yet.')

In [ ]:
# 3. Apply explicit FP16 compatibility changes to this temporary pinned checkout.
# Upstream uses BF16; changing the YAML alone is not sufficient on a T4.
source_path = REPO / 'ltx_video/inference.py'
source = source_path.read_text()
changes = [
    ('    elif precision == "bfloat16":',
     '    elif precision == "float16":\n        return Transformer3DModel.from_pretrained(ckpt_path).to(torch.float16)\n    elif precision == "bfloat16":'),
    ('    transformer = transformer.to(device)\n    vae = vae.to(device)\n    text_encoder = text_encoder.to(device)',
     '    component_dtype = torch.float16 if precision == "float16" else torch.bfloat16\n    transformer = transformer.to(device)\n    vae = vae.to(dtype=component_dtype).to(device)\n    text_encoder = text_encoder.to(dtype=component_dtype).to(device)'),
    ('    vae = vae.to(torch.bfloat16)\n    text_encoder = text_encoder.to(torch.bfloat16)',
     '    vae = vae.to(component_dtype)\n    text_encoder = text_encoder.to(component_dtype)'),
]
for old, new in changes:
    if old not in source and new in source:
        continue
    if source.count(old) != 1:
        raise RuntimeError('Pinned source does not match the expected precision patch. Stop rather than guessing.')
    source = source.replace(old, new, 1)
compile(source, str(source_path), 'exec')
source_path.write_text(source)
print('FP16 loader prepared. Its numerical quality and memory use still need a real GPU test.')

In [ ]:
# 4. Download only pinned required components; disable extra prompt-enhancement models.
setup = r'''
import json
from pathlib import Path
import torch, yaml
from huggingface_hub import hf_hub_download, snapshot_download

root = Path('/content/ltx-video-test')
model_revision = '8984fa25007f376c1a299016d0957a37a2f797bb'
text_revision = 'b89adadeccd9ead2adcb9fa2825d3fabec48d404'
if not torch.cuda.is_available():
    raise RuntimeError('The isolated environment cannot access the GPU.')
print('Torch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0), flush=True)
checkpoint = hf_hub_download('Lightricks/LTX-Video', 'ltxv-2b-0.9.8-distilled.safetensors', revision=model_revision)
upscaler = hf_hub_download('Lightricks/LTX-Video', 'ltxv-spatial-upscaler-0.9.8.safetensors', revision=model_revision)
text_root = snapshot_download('PixArt-alpha/PixArt-XL-2-1024-MS', revision=text_revision, allow_patterns=['text_encoder/*', 'tokenizer/*'])
config = yaml.safe_load((root / 'LTX-Video/configs/ltxv-2b-0.9.8-distilled.yaml').read_text())
config.update(checkpoint_path=checkpoint, spatial_upscaler_model_path=upscaler,
    text_encoder_model_name_or_path=text_root, precision='float16', prompt_enhancement_words_threshold=0)
(root / 'test-config.yaml').write_text(yaml.safe_dump(config, sort_keys=False))
(root / 'revisions.json').write_text(json.dumps({'model': model_revision, 'text_encoder': text_revision}, indent=2))
print('Pinned components ready.', flush=True)
'''
(ROOT / 'setup_models.py').write_text(setup)
run([PYTHON, ROOT / 'setup_models.py'])

In [ ]:
# Write the shared single-shot runner used to generate one take per call.
runner = r'''
import json, sys, time
from pathlib import Path
import torch
from ltx_video.inference import InferenceConfig, infer

settings = json.loads(Path(sys.argv[1]).read_text())
torch.cuda.reset_peak_memory_stats()
started = time.monotonic()
try:
    infer(InferenceConfig(**settings))
except Exception as error:
    print('TEST STOPPED:', type(error).__name__, str(error), flush=True)
    raise
finally:
    report = {'generation_seconds': time.monotonic()-started,
        'peak_gpu_allocated_gib': torch.cuda.max_memory_allocated()/1024**3,
        'peak_gpu_reserved_gib': torch.cuda.max_memory_reserved()/1024**3,
        'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__}
    Path(settings['output_path'], 'measurement.json').write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2), flush=True)
'''
(ROOT / 'run_shot.py').write_text(runner)
print('Runner ready.')

In [ ]:
# 5. Install the small web server pieces in Colab's own Python (not the model env).
run([sys.executable, '-m', 'pip', 'install', '--quiet', 'fastapi==0.115.12', 'uvicorn==0.34.2', 'pydantic==2.11.4'])
cf = ROOT / 'cloudflared'
if not cf.exists():
    run(['curl', '-fsSL', '-o', str(cf), 'https://github.com/cloudflare/cloudflared/releases/download/2025.4.0/cloudflared-linux-amd64'])
    run(['chmod', '+x', str(cf)])
print('Server dependencies ready.')

In [ ]:
# 6. Write the generation server. One job at a time; results stay in the session.
server = r'''
import json, os, queue, secrets, subprocess, threading, time, urllib.request, uuid
from pathlib import Path

from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

ROOT = Path('/content/ltx-video-test')
REPO = ROOT / 'LTX-Video'
PYTHON = ROOT / 'env/bin/python'
TOKEN = os.environ['LTX_SERVER_TOKEN']

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

jobs = {}
work = queue.Queue()

class Shot(BaseModel):
    prompt: str
    width: int = 384
    height: int = 256
    frames: int = 33
    fps: int = 24
    seed: int = 42
    image_url: str | None = None

def check(token):
    if token != TOKEN:
        raise HTTPException(status_code=401, detail='Bad token')

def validate(s):
    if not s.prompt.strip():
        raise HTTPException(400, 'Enter a shot prompt.')
    if s.width % 32 or s.height % 32 or min(s.width, s.height) < 128:
        raise HTTPException(400, 'Dimensions must be divisible by 32 and at least 128.')
    if s.width * s.height > 576 * 320:
        raise HTTPException(400, 'Area above the tested memory budget (max 576x320).')
    if s.frames < 9 or s.frames > 97 or (s.frames - 1) % 8:
        raise HTTPException(400, 'Frames must be 8n+1, between 9 and 97.')
    if not 8 <= s.fps <= 30:
        raise HTTPException(400, 'FPS must be between 8 and 30.')

def worker():
    while True:
        job_id, s = work.get()
        job = jobs[job_id]
        job['status'] = 'running'
        out = ROOT / 'outputs' / job_id
        out.mkdir(parents=True, exist_ok=True)
        try:
            settings = dict(prompt=s.prompt, width=s.width, height=s.height, num_frames=s.frames,
                frame_rate=s.fps, seed=s.seed, offload_to_cpu=True,
                pipeline_config=str(ROOT / 'test-config.yaml'), output_path=str(out))
            if s.image_url:
                img = out / 'start-image'
                with urllib.request.urlopen(s.image_url, timeout=30) as r:
                    img.write_bytes(r.read())
                settings.update(conditioning_media_paths=[str(img)], conditioning_start_frames=[0], conditioning_strengths=[1.0])
            (out / 'request.json').write_text(json.dumps(settings, indent=2))
            env = dict(os.environ, PYTHONPATH=str(REPO), PYTHONUNBUFFERED='1', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
            started = time.monotonic()
            subprocess.run([str(PYTHON), str(ROOT / 'run_shot.py'), str(out / 'request.json')],
                           check=True, env=env, cwd=REPO, capture_output=True, text=True, timeout=1800)
            videos = list(out.glob('*.mp4'))
            if len(videos) != 1:
                raise RuntimeError('Expected exactly one MP4 from the runner.')
            job.update(status='done', path=str(videos[0]), seconds=round(time.monotonic() - started, 1))
        except Exception as e:
            job.update(status='error', error=f'{type(e).__name__}: {e}'[:500])

threading.Thread(target=worker, daemon=True).start()

@app.get('/health')
def health(x_token: str = Header(default='')):
    check(x_token)
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
    return {'ok': True, 'gpu': gpu, 'queued': work.qsize()}

@app.post('/generate')
def generate(s: Shot, x_token: str = Header(default='')):
    check(x_token)
    validate(s)
    job_id = uuid.uuid4().hex[:12]
    jobs[job_id] = {'status': 'queued', 'created': time.time()}
    work.put((job_id, s))
    return {'job_id': job_id}

@app.get('/jobs/{job_id}')
def status(job_id: str, x_token: str = Header(default='')):
    check(x_token)
    job = jobs.get(job_id)
    if not job:
        raise HTTPException(404, 'Unknown job')
    return {k: v for k, v in job.items() if k != 'path'}

@app.get('/jobs/{job_id}/video')
def video(job_id: str, x_token: str = Header(default='')):
    check(x_token)
    job = jobs.get(job_id)
    if not job or job.get('status') != 'done':
        raise HTTPException(409, 'Video is not ready')
    return FileResponse(job['path'], media_type='video/mp4', filename=f'ltx-{job_id}.mp4')
'''
(ROOT / 'server.py').write_text(server)
print('Server written.')

In [ ]:
# 7. Start the server and tunnel, then print the address and token for the app.
import re, secrets, threading, time
os.environ['LTX_SERVER_TOKEN'] = secrets.token_urlsafe(18)

def serve():
    import uvicorn
    uvicorn.run('server:app', host='127.0.0.1', port=8734, log_level='warning')

sys.path.insert(0, str(ROOT))
threading.Thread(target=serve, daemon=True).start()
time.sleep(3)

tunnel = subprocess.Popen([str(ROOT / 'cloudflared'), 'tunnel', '--url', 'http://127.0.0.1:8734', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break
if not public_url:
    raise RuntimeError('The tunnel did not start. Re-run this cell once; if it repeats, stop.')
print('=' * 60)
print('Paste these into the Video page in the app:')
print('Endpoint:', public_url)
print('Token:   ', os.environ['LTX_SERVER_TOKEN'])
print('=' * 60)
print('Keep this tab open while generating. Closing it ends the address and token.')